# Brady-GPT: a model that talks like me in class

OLMo 3 7B, retrained on my class recordings, compared with the model it started from.

**To run:** use a GPU (a T4 works, at 8-bit), add `HF_TOKEN` under the key icon (Secrets), choose **Runtime > Run all**, and pick `demo_data.json` when asked. Allow about 30 minutes the first time, so start before class.

Nothing run here is saved with this notebook, and the class text stays in this session.

In [ ]:
%pip install -q transformers==5.21.0 accelerate bitsandbytes

In [ ]:
import gc, html, json, random, re, statistics, time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
from google.colab import files, userdata
from huggingface_hub import HfApi
from IPython.display import HTML, display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer

transformers.logging.set_verbosity_error()

# The private class data comes from my Mac, never from GitHub.
print("Choose demo_data.json (ai_lab/experiments/brady-gpt/data/demo/ on my Mac)")
uploaded = files.upload()
assert uploaded, "No file chosen: run this cell again."
data = json.loads(next(iter(uploaded.values())))

try:
    login_token = userdata.get("HF_TOKEN")  # my model is private: this login is the only way to load it
except Exception:
    login_token = None
HAS_GPU = torch.cuda.is_available()
USE_MODELS = bool(HAS_GPU and login_token)
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none")
BASE, MINE = "allenai/Olmo-3-1025-7B", "bradyrutherford/minerva-olmo3-7b-B"
if login_token:  # checks the login without ever printing it
    try:
        HfApi(token=login_token).model_info(MINE)
        print("HF_TOKEN: can read my private model")
    except Exception as error:
        USE_MODELS = False
        print("HF_TOKEN: found, but it cannot read my private model:", type(error).__name__)
else:
    print("HF_TOKEN: not found. The secret must be named exactly HF_TOKEN, with notebook access on.")
print("Live models:", "on" if USE_MODELS else "OFF. Needs a T4 and HF_TOKEN. Saved answers still work.")

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.edgecolor": "#d9d8d3", "grid.color": "#e8e7e3", "grid.linewidth": 1, "axes.grid": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 12, "axes.titlesize": 13,
    "axes.titlelocation": "left", "lines.linewidth": 2, "lines.solid_capstyle": "round",
})
BLUE, GRAY = "#2a78d6", "#a3a29c"


def card(title, rows):
    """One moment on screen: a heading, then labelled answers."""
    body = "".join(
        f"<div style='margin:12px 0'><b>{html.escape(label)}</b><br>{html.escape(text)}</div>" for label, text in rows
    )
    display(HTML(f"<div style='font-size:16px;line-height:1.5;max-width:900px;padding:8px 16px;"
                 f"border-left:3px solid {BLUE}'><h4 style='margin:4px 0'>{html.escape(title)}</h4>{body}</div>"))


def short(text, words=60):
    parts = text.split()
    return text if len(parts) <= words else "... " + " ".join(parts[-words:])

## 1. How I trained it

| Step | What it did | Why I chose it |
|---|---|---|
| **1. Data** | Turned 477 class recordings into 2,950 of my own examples and 15,588 of my classmates', with names masked. 94 classes were held back for testing. | Class recordings are the only record of how I actually speak. |
| **2. Base model** | Started from OLMo 3 7B (7.3 billion weights). | It's fully open, and small enough for one GPU. |
| **3. SFT** | Trained a small add-on (0.54% of the weights) on my examples 3x and my classmates' 1x, each tagged with who is speaking. | Classmates add class vocabulary and the tag keeps the voices apart. It beat a me-only version on held-out answers (loss 2.572 vs 2.586). |
| **4. DPO** | Showed it 2,445 pairs: my real answer against its own attempt at the same question, and pushed it toward mine. | SFT predicts words, not style. DPO cost a little loss (2.572 to 2.586), but in blind tests I picked it over SFT 7-4 and over the me-only DPO 6-3. |
| **5. Safety checks** | Checked it doesn't repeat what it read: 0 of 3,944 answers repeated someone else's words. | It trained on my classmates' speech and must not recite it. |
| **6. Context test** | Same model, no retraining: gave it more of the class in its prompt. | A class bot has to follow the discussion, and this needed no new training. |

In [ ]:
# Real numbers from the training logs and the context test. No class text.
# Step 3: train_sft.py --variant B
SFT_PROGRESS = [0.07, 0.13, 0.2, 0.26, 0.33, 0.39, 0.46, 0.52, 0.59, 0.65, 0.72, 0.78, 0.85, 0.91, 0.98, 1.0]
SFT_LOSS = [2.741, 2.704, 2.689, 2.681, 2.675, 2.672, 2.665, 2.663, 2.661, 2.661, 2.659, 2.658, 2.657, 2.657, 2.657, 2.657]
# Step 4: train_dpo.py --variant B --spin <its own answers> --loss sigmoid_norm --sft-weight 1
DPO_PROGRESS = [0.03, 0.07, 0.1, 0.13, 0.16, 0.2, 0.23, 0.26, 0.29, 0.33, 0.36, 0.39, 0.43, 0.46, 0.49, 0.52, 0.56, 0.59, 0.62, 0.65, 0.69, 0.72, 0.75, 0.79, 0.82, 0.85, 0.88, 0.92, 0.95, 0.98]
DPO_PREFERS_MINE = [22, 50, 60, 61, 57, 46, 44, 46, 61, 54, 57, 59, 54, 59, 65, 57, 66, 72, 49, 59, 59, 60, 51, 48, 45, 62, 64, 65, 56, 59]
# Step 6: eval_context.py, 618 held-out answers
CONTEXT = {"Trained\nprompt": 2.609, "+ last 2,500\ntokens": 2.528, "+ whole\nclass": 2.515}

fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

a.plot(SFT_PROGRESS, SFT_LOSS, color=BLUE)
a.set(title="Step 3, SFT: loss on held-out answers", xlabel="Share of training done", ylabel="Loss (lower is better)")
a.annotate(f"{SFT_LOSS[0]:.3f}", (SFT_PROGRESS[0], SFT_LOSS[0]), xytext=(8, 0), textcoords="offset points")
a.annotate(f"{SFT_LOSS[-1]:.3f}", (SFT_PROGRESS[-1], SFT_LOSS[-1]), xytext=(-10, 10), textcoords="offset points", ha="right")

b.axhline(50, color="#52514e", linewidth=1)
b.text(0.45, 47, "coin flip", ha="center", va="top", color="#52514e")
b.plot(DPO_PROGRESS, DPO_PREFERS_MINE, color=BLUE)
b.set(title="Step 4, DPO: prefers my answer to its own", xlabel="Share of training done", ylabel="Pairs (%)", ylim=(0, 100))

names, values = list(CONTEXT), list(CONTEXT.values())
c.plot(names, values, color=BLUE, marker="o", markersize=8)
c.set(title="Step 6, context: loss on 618 held-out answers", ylabel="Loss (lower is better)", ylim=(2.48, 2.64), xlim=(-0.3, 2.3))
for name, value in zip(names, values):
    c.annotate(f"{value:.3f}", (name, value), xytext=(0, 12), textcoords="offset points", ha="center")
plt.show()

## 2. Same question: my model, the base model and Botnik

- **Base:** OLMo 3 7B before any of my training.
- **My model:** the result of the steps above.
- **Botnik:** a predictive-text keyboard, like Botnik Studios'. It counts which word follows which in my answers and never reads the question.

Base and my model get the exact same prompt, settings and precision.

In [ ]:
# Botnik: the next word is drawn from what I said after the last two words. That's the whole model.
next_word = defaultdict(Counter)
for text in data["my_answers"]:
    words = ["<s>", "<s>"] + text.split() + ["</s>"]
    for w1, w2, w3 in zip(words, words[1:], words[2:]):
        next_word[w1, w2][w3] += 1


def botnik(seed=0, max_words=150):
    rng = random.Random(seed)
    w1, w2, out = "<s>", "<s>", []
    while len(out) < max_words:
        options = next_word[w1, w2]
        w3 = rng.choices(list(options), list(options.values()))[0]
        if w3 == "</s>":
            break
        out.append(w3)
        w1, w2 = w2, w3
    return " ".join(out)


print(f"Botnik learned {sum(map(len, next_word.values())):,} word patterns from {len(data['my_answers']):,} answers.\n")
print(botnik(seed=1))

In [ ]:
# Only one 8-bit model fits on a T4 at a time, so the base goes first and my model stays loaded.
STOPS = ["\nProfessor:", "\nStudent ", "\nSpeaker:", "\nBrady:"]  # it may never speak as someone else


def load(repo):
    start = time.time()
    # Full precision needs about 15 GB, which is all a T4 has, so a T4 gets 8-bit and bigger GPUs get the real thing.
    full = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 20
    settings = {"dtype": torch.bfloat16} if full else {
        "dtype": torch.float16, "quantization_config": BitsAndBytesConfig(load_in_8bit=True)}
    model = AutoModelForCausalLM.from_pretrained(repo, device_map={"": 0}, token=login_token, **settings).eval()
    print(f"{'Base model' if repo == BASE else 'My model'}: {'full precision' if full else '8-bit'}, loaded in "
          f"{time.time() - start:.0f}s, {torch.cuda.memory_allocated() / 1e9:.1f} GB on the GPU")
    return model


def free_gpu():  # after `del`-ing every name for the model
    gc.collect()
    torch.cuda.empty_cache()


def to_prompt(content):
    return f"<|user|>\n{content}\n<|assistant|>\n"  # the layout it trained on


@torch.inference_mode()
def answer(model, content, seed=0, stream=False):
    ids = tok(to_prompt(content), return_tensors="pt", add_special_tokens=False).to(model.device)
    torch.manual_seed(seed)
    out = model.generate(**ids, do_sample=True, temperature=0.8, top_p=0.95, max_new_tokens=256,
                         eos_token_id=tok.eos_token_id, stop_strings=STOPS, tokenizer=tok,
                         streamer=TextStreamer(tok, skip_prompt=True, skip_special_tokens=True) if stream else None)
    text = tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)
    for stop in STOPS:
        text = text.split(stop)[0]
    return text.strip()


@torch.inference_mode()
def loss_on(model, content, real):
    """How surprised the model is by my real words: lower means closer to how I talk."""
    prompt = tok(to_prompt(content), add_special_tokens=False).input_ids
    reply = tok(real + tok.eos_token, add_special_tokens=False).input_ids
    ids = torch.tensor([prompt + reply]).to(model.device)
    logits = model(ids).logits[0, len(prompt) - 1:-1].float()
    return torch.nn.functional.cross_entropy(logits, ids[0, len(prompt):]).item()


results = {}
if USE_MODELS:
    tok = AutoTokenizer.from_pretrained(MINE, token=login_token)
    rows = data["scorecard"]
    for label, repo in [("Base", BASE), ("My model", MINE)]:
        model = load(repo)
        start = time.time()
        results[label] = {
            "answers": [answer(model, r["prompt"], seed=i) for i, r in enumerate(rows)],
            "losses": [loss_on(model, r["prompt"], r["real"]) for r in rows],
        }
        print(f"{label}: {len(rows)} answers in {time.time() - start:.0f}s")
        if label == "Base":
            del model
            free_gpu()
    mine = model
    del model
    results["My model"]["moments"] = [answer(mine, m["prompt"], seed=i) for i, m in enumerate(data["moments"])]
else:
    print("Skipped: needs a T4 and HF_TOKEN.")

In [ ]:
# Three of those questions, side by side
if results:
    for i, row in enumerate(data["scorecard"][:3]):
        card(row["course"].split(" - ")[0], [
            ("Professor", short(row["ask"])),
            ("What I said", row["real"]),
            ("My model", results["My model"]["answers"][i]),
            ("Base model", results["Base"]["answers"][i]),
            ("Botnik", botnik(seed=i)),
        ])
else:
    print("Skipped: needs a T4 and HF_TOKEN.")

In [ ]:
# The scorecard: the same held-out questions for everyone.
WORD = re.compile(r"[a-z]+(?:'[a-z]+)?")
FILLER = {"like", "yeah", "yep", "um", "uh", "okay", "ok", "kinda", "gonna", "wanna", "really",
          "just", "thing", "things", "think", "know", "mean", "sort", "kind", "lot"}


def n_words(text):
    return len(WORD.findall(text.lower()))


def per_1000(pattern, texts):
    return 1000 * sum(len(re.findall(pattern, t, re.I)) for t in texts) / max(1, sum(map(n_words, texts)))


def content(text):
    return Counter(w for w in WORD.findall(text.lower())
                   if w not in ENGLISH_STOP_WORDS and w not in FILLER and len(w) > 2)


def overlap(a, b):  # share of the meaningful words two answers have in common
    a, b = content(a), content(b)
    total = sum(a.values()) + sum(b.values())
    return 2 * sum((a & b).values()) / total if total else 0.0


if results:
    real = [r["real"] for r in data["scorecard"]]
    answers = {"Me": real, "My model": results["My model"]["answers"], "Base": results["Base"]["answers"],
               "Botnik": [botnik(seed=i) for i in range(len(real))]}
    table = pd.DataFrame({
        name: {
            "Median words": statistics.median(map(n_words, texts)),
            '"like" per 1,000 words': per_1000(r"\blike\b", texts),
            '"I think" per 1,000 words': per_1000(r"\bi think\b", texts),
            "Word overlap with my real answer": None if name == "Me" else statistics.mean(map(overlap, texts, real)),
            "Loss on my real words": statistics.mean(results[name]["losses"]) if name in results else None,
        } for name, texts in answers.items()
    }).T
    display(table.round(3).astype(object).where(table.notna(), "n/a"))

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.2), constrained_layout=True)
    names = ["My model", "Base", "Botnik"]
    for ax, column in zip(axes, ['Median words', '"like" per 1,000 words', 'Word overlap with my real answer']):
        values = table.loc[names, column].astype(float)
        bars = ax.barh(names, values, height=0.5, color=[BLUE, GRAY, GRAY])
        ax.bar_label(bars, fmt="%.2f" if "overlap" in column else "%.0f", padding=4,
                     bbox=dict(facecolor="#fcfcfb", edgecolor="none", pad=1))
        ax.set_title(column, pad=20)
        ax.set_axisbelow(True)
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        ax.margins(x=0.15)
        if column != 'Word overlap with my real answer':  # no line for overlap: I'd match myself exactly
            ax.axvline(table.loc["Me", column], color="#0b0b0b", linewidth=1)
            ax.annotate("Me", (table.loc["Me", column], 1.02), xycoords=("data", "axes fraction"), ha="center")
    plt.show()

    full = statistics.mean(r["full_loss"] for r in data["scorecard"])
    print(f"Precision check, same {len(real)} answers: my model's loss is {table.loc['My model', 'Loss on my real words']:.3f} here, "
          f"{full:.3f} in the earlier full-precision run.")
    print(f"This scorecard uses {len(real)} questions. The context test used 618.")
else:
    print("Skipped: needs a T4 and HF_TOKEN.")

## 3. Five moments from my classes

All from classes held back from training, so it never saw them. They were picked at random, one per course, with no hand-picking of good answers.

- **Saved answer:** from the full-precision model on a 96 GB GPU, reading the whole class so far.
- **Live answer:** from the GPU running this notebook, reading the last 2,500 tokens of the class.

In [ ]:
def show_moment(i):
    m = data["moments"][i]
    rows = [("Professor", short(m["ask"])), ("What I said", m["real"]), ("My model (saved answer)", m["saved"])]
    if results:
        rows.append(("My model (live)", results["My model"]["moments"][i]))
    card(f"Moment {i + 1}: {m['course']}", rows)

In [ ]:
show_moment(0)

In [ ]:
show_moment(1)

In [ ]:
show_moment(2)

In [ ]:
show_moment(3)

In [ ]:
show_moment(4)

## 4. Ask it anything, live

Type a question and run the cell. My model answers straight away. Ticking `also_base` swaps the models on the GPU, which takes a few minutes each way.

In [ ]:
question = "If this model can sound like me, what's still missing before it could really join a class discussion?"  #@param {type:"string"}
class_name = "CS156 Finding Patterns in Data with Machine Learning - Class presentations"  #@param {type:"string"}
so_far = ""  #@param {type:"string"}
also_base = False  #@param {type:"boolean"}

history = f"Class so far:\n{so_far}\n\n" if so_far.strip() else ""
live = f"Answer as: Brady\nClass discussion in {class_name}\n\n{history}Professor: {question}\n\nWhat do you say?"

if USE_MODELS:
    print("My model:\n")
    answer(mine, live, seed=random.randrange(10**6), stream=True)
    if also_base:
        del mine
        free_gpu()
        base = load(BASE)
        print("\nBase model:\n")
        answer(base, live, seed=random.randrange(10**6), stream=True)
        del base
        free_gpu()
        mine = load(MINE)
else:
    print("Skipped: needs a T4 and HF_TOKEN.")

## After class

Tick `wipe` and run this cell. It deletes the session, including the private model and the class data.

In [ ]:
wipe = False  #@param {type:"boolean"}
if wipe:
    from google.colab import runtime
    runtime.unassign()